# Inteligência Artificial — AutoML: aula interativa
### Versão do aluno — sem gabarito
Execute os experimentos prontos, responda às 15 perguntas e entregue o arquivo JSON. A correção será feita pelo professor.


# PARTE 1 — Entendendo o problema de IA
## 1.1 LEIA — Como a máquina aprende a classificar?

Imagine uma tabela: cada **linha** é uma observação, cada **coluna de entrada** é uma característica e a **classe-alvo** indica a resposta conhecida. Na aprendizagem supervisionada, um algoritmo observa exemplos rotulados para aprender a prever a classe de novos exemplos.

Nesta atividade, existem **duas classes**. Portanto, o problema é de **classificação binária**. As entradas são 30 medidas numéricas; o conjunto contém 569 observações.

**Termos essenciais:**
- **Amostra:** uma linha da tabela.
- **Variável preditora (feature):** uma medida usada como entrada.
- **Rótulo (target):** a classe que queremos prever.
- **Modelo:** regra aprendida a partir dos exemplos.

## 1.2 EXECUTE — Preparar o ambiente
A célula a seguir instala e importa as bibliotecas necessárias. **Execute uma vez e aguarde a mensagem de confirmação.**

In [ ]:
# Preparação: execute uma única vez
%pip -q install "flaml[automl]>=2.3,<3" ipywidgets
import numpy as np, pandas as pd, matplotlib.pyplot as plt, ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from flaml import AutoML
import json, datetime

# Formulário sem gabarito: as respostas são exportadas para o professor.
RESPOSTAS = {}
def questionario(secao, perguntas):
    itens=[]
    for p in perguntas:
        op=widgets.RadioButtons(options=[('Selecione uma resposta',None)]+[(x,x) for x in p['opcoes']],value=None,layout=widgets.Layout(width='auto'))
        itens.append((p,op))
    saida=widgets.Output()
    def salvar(_):
        with saida:
            clear_output()
            for p,op in itens:
                RESPOSTAS[p['id']]=op.value
            print('Respostas registradas nesta seção. Você pode alterá-las e salvar novamente.')
    botao=widgets.Button(description='Salvar respostas desta seção',button_style='info',icon='save')
    botao.on_click(salvar)
    display(widgets.VBox([widgets.HTML(f'<h3>{secao}</h3>')]+[widgets.VBox([widgets.HTML(f'<b>{p["id"]}.</b> {p["texto"]} <i>({p["pontos"]:.1f} ponto)</i>'),op]) for p,op in itens]+[botao,saida]))
print('Ambiente preparado! Continue para a próxima célula.')


## 1.3 EXECUTE E OBSERVE — Conheça os dados
Clique em ▶ na próxima célula. Observe o **número de amostras**, o **número de características** e a distribuição das classes.

**Pergunta para pensar:** se uma tabela contém 569 linhas e 30 características, qual número representa a quantidade de informações de entrada de cada observação?

In [ ]:
dados=load_breast_cancer(as_frame=True)
X=dados.data.copy(); y=dados.target.copy()
print('Amostras:',X.shape[0]);print('Variáveis preditoras:',X.shape[1]);print('Classes:',dict(enumerate(dados.target_names)))
display(X.head(5))
contagens=y.value_counts().sort_index()
ax=contagens.plot.bar(rot=0,title='Quantidade de amostras por classe')
ax.set_xticklabels([f'{i}: {dados.target_names[i]}' for i in contagens.index]);ax.set_ylabel('Amostras');plt.show()


## 1.4 RESPONDA E CORRIJA — Perguntas sobre os dados
Escolha uma alternativa em cada questão. Clique em **Salvar respostas desta seção** e confira se todas as respostas foram registradas.

**Conceito avaliado:** identificar amostras, variáveis e tipo de aprendizagem.

In [ ]:
questionario('Seção 1 — Dados',[
 {'id':'Q1','texto':'O que representa cada linha do dataset?','opcoes':['Uma amostra com suas medidas','Um algoritmo de classificação','Uma métrica de avaliação'],'pontos':0.5},
 {'id':'Q2','texto':'Qual é o tipo de tarefa de aprendizagem de máquina?','opcoes':['Regressão de preços','Classificação binária','Agrupamento sem rótulos'],'pontos':0.5},
 {'id':'Q3','texto':'Quantas variáveis preditoras existem?','opcoes':['2','30','569'],'pontos':0.5}
])


# PARTE 2 — Construindo uma referência para comparação
## 2.1 LEIA — Por que não avaliar no mesmo dado usado para treinar?

Um modelo pode memorizar padrões dos exemplos vistos. Para verificar se ele **generaliza**, separamos dados que ele não verá durante o treinamento.

- **Treino:** usado para ajustar o modelo.
- **Validação:** usada para comparar configurações durante a seleção de modelos (na Parte 3, o AutoML fará isso internamente).
- **Teste:** reservado para a avaliação final.

Primeiro usaremos uma **regressão logística**, um algoritmo clássico de classificação, como **modelo de referência (baseline)**. Embora tenha o nome “regressão”, aqui ela prevê classes.

## 2.2 EXECUTE E OBSERVE
A célula seguinte divide os dados, treina o modelo e apresenta métricas e matriz de confusão. Você não precisa modificar nada.

**Acurácia:** fração de previsões corretas.  
**Matriz de confusão:** tabela que separa acertos e erros para cada classe.

**Atenção:** uma acurácia alta não significa que todos os tipos de erro sejam igualmente aceitáveis.

In [ ]:
X_treino,X_teste,y_treino,y_teste=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
baseline=make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,random_state=42))
baseline.fit(X_treino,y_treino)
pred_base=baseline.predict(X_teste)
acc_base=accuracy_score(y_teste,pred_base)
bal_base=balanced_accuracy_score(y_teste,pred_base)
f1_base=f1_score(y_teste,pred_base,average='macro')
print('Treino:',len(y_treino),'amostras | Teste:',len(y_teste),'amostras')
print(f'Acurácia: {acc_base:.3f} | Acurácia balanceada: {bal_base:.3f} | F1 macro: {f1_base:.3f}')
ConfusionMatrixDisplay.from_predictions(y_teste,pred_base,display_labels=dados.target_names,cmap='Blues');plt.title('Matriz de confusão — modelo convencional');plt.show()


## 2.3 RESPONDA E CORRIJA — Treino, teste e métricas
Use o que você observou no experimento para responder. Pense especialmente em **generalização** e **erros por classe**.

In [ ]:
questionario('Seção 2 — Modelo convencional',[
 {'id':'Q4','texto':'Para que serve o conjunto de teste?','opcoes':['Para escolher os hiperparâmetros durante o treino','Para estimar o desempenho em dados não vistos','Para aumentar o número de exemplos de treino'],'pontos':0.5},
 {'id':'Q5','texto':'O que mede a acurácia?','opcoes':['Proporção total de previsões corretas','Tempo gasto no treinamento','Quantidade de variáveis do dataset'],'pontos':0.5},
 {'id':'Q6','texto':'Por que utilizar a matriz de confusão?','opcoes':['Para mostrar somente a velocidade','Para observar erros e acertos por classe','Para substituir o conjunto de teste'],'pontos':0.5}
])


# PARTE 3 — AutoML: automatizando escolhas
## 3.1 LEIA — O que muda em relação ao modelo anterior?

No experimento anterior, o algoritmo foi escolhido previamente: **regressão logística**. Agora, o **AutoML (Automated Machine Learning)** tentará encontrar automaticamente uma boa configuração de modelo.

Um processo de AutoML pode automatizar a **seleção de algoritmos** e o **ajuste de hiperparâmetros** (configurações definidas antes do treinamento). Isso **não** elimina a necessidade de definir o problema, verificar os dados e avaliar os resultados criticamente.

**Biblioteca utilizada:** FLAML.  
**Orçamento de busca:** cerca de 45 segundos.  
**Critério de seleção:** acurácia estimada por validação cruzada no conjunto de treino.

## 3.2 EXECUTE E OBSERVE — Busca automática
Clique em ▶ e aguarde. A execução pode levar mais tempo que as anteriores.

Observe o **algoritmo selecionado** e a **acurácia no conjunto de teste**. O algoritmo selecionado pode mudar dependendo do ambiente e do orçamento de busca.

**Importante:** o conjunto de teste não deve ser utilizado para escolher hiperparâmetros.

In [ ]:
automl=AutoML()
config={'time_budget':45,'task':'classification','metric':'accuracy','eval_method':'cv','n_splits':3,'seed':42,'verbose':0,'n_jobs':1,'estimator_list':['lrl1','rf','extra_tree']}
try:
    automl.fit(X_train=X_treino,y_train=y_treino,**config)
    pred_auto=automl.predict(X_teste)
    acc_auto=accuracy_score(y_teste,pred_auto)
    bal_auto=balanced_accuracy_score(y_teste,pred_auto)
    f1_auto=f1_score(y_teste,pred_auto,average='macro')
    MODELO_OK=True
    print('Algoritmo selecionado:',automl.best_estimator)
    print('Melhores hiperparâmetros:',automl.best_config)
    print(f'Acurácia no teste: {acc_auto:.3f} | Acurácia balanceada: {bal_auto:.3f} | F1 macro: {f1_auto:.3f}')
except Exception as e:
    MODELO_OK=False
    print('Não foi possível concluir a busca:',str(e))
    print('Tente executar esta célula novamente. Se persistir, reinicie o ambiente e execute tudo em ordem.')


## 3.3 RESPONDA E CORRIJA — Como funciona o AutoML?
Responda com base no processo executado e nas diferenças entre **treinar**, **selecionar** e **testar** um modelo.

In [ ]:
questionario('Seção 3 — AutoML',[
 {'id':'Q7','texto':'Qual tarefa o AutoML executou neste experimento?','opcoes':['Seleção automática de algoritmo e hiperparâmetros','Coleta automática de novos pacientes','Substituição da avaliação no teste'],'pontos':0.5},
 {'id':'Q8','texto':'Qual conjunto foi utilizado na busca do AutoML?','opcoes':['Somente o conjunto de teste','O conjunto de treino, com validação interna','O conjunto de teste e treino sem separação'],'pontos':0.5},
 {'id':'Q9','texto':'O algoritmo escolhido pelo FLAML será obrigatoriamente o mesmo em qualquer computador?','opcoes':['Sim, sempre','Não, orçamento e ambiente podem influenciar','Sim, porque há somente um algoritmo disponível'],'pontos':0.5},
 {'id':'Q10','texto':'O que significa time_budget=45?','opcoes':['O AutoML dispõe de cerca de 45 segundos para a busca','O dataset tem 45 amostras','O modelo fará exatamente 45 previsões'],'pontos':0.5}
])


# PARTE 4 — Comparando duas abordagens de IA
## 4.1 LEIA — AutoML é sempre melhor?

**Não necessariamente.** A busca automática pode encontrar um modelo melhor, equivalente ou pior que o modelo de referência no teste. Além disso, uma diferença pequena pode depender da divisão dos dados.

A comparação deve usar o **mesmo conjunto de teste** e métricas calculadas da mesma forma.

## 4.2 EXECUTE E OBSERVE — Comparação lado a lado
A próxima célula mostra os resultados. Preste atenção às **acurácias dos dois modelos**: uma das perguntas será corrigida automaticamente usando os valores que aparecerem na sua execução.

In [ ]:
if not globals().get('MODELO_OK',False):
    print('Execute primeiro a busca AutoML da Parte 3.')
else:
    comparacao=pd.DataFrame({'Modelo':['Convencional','AutoML'],'Acurácia':[acc_base,acc_auto],'Acurácia balanceada':[bal_base,bal_auto],'F1 macro':[f1_base,f1_auto]})
    display(comparacao.round(3))
    comparacao.set_index('Modelo')[['Acurácia','Acurácia balanceada','F1 macro']].plot.bar(rot=0,ylim=(0,1.05),figsize=(8,4));plt.ylabel('Pontuação');plt.title('Comparação no mesmo conjunto de teste');plt.show()
    ConfusionMatrixDisplay.from_predictions(y_teste,pred_auto,display_labels=dados.target_names,cmap='Greens');plt.title('Matriz de confusão — AutoML');plt.show()


## 4.3 RESPONDA E CORRIJA — Interprete os seus resultados
**A questão 11 depende da sua execução:** leia a tabela de comparação antes de responder. Não há obrigação de o AutoML vencer.

As demais questões verificam se você compreendeu os cuidados necessários para uma comparação justa.

In [ ]:
if not globals().get('MODELO_OK',False):
    print('Execute primeiro a Parte 3 e depois esta célula.')
else:
    questionario('Seção 4 — Comparação',[
     {'id':'Q11','texto':'Qual modelo apresentou maior acurácia no conjunto de teste da SUA execução?','opcoes':['AutoML','Convencional','Empate'],'pontos':2.0},
     {'id':'Q12','texto':'Uma diferença pequena de acurácia em um único teste permite afirmar superioridade universal?','opcoes':['Sim, sempre','Não; é preciso considerar incerteza, contexto e validação adicional','Sim, se o modelo for AutoML'],'pontos':0.5},
     {'id':'Q13','texto':'Por que comparar os dois modelos no mesmo conjunto de teste?','opcoes':['Para garantir condições de avaliação comparáveis','Para fazer o AutoML aprender com as respostas do teste','Para eliminar todos os erros'],'pontos':0.5}
    ])


# PARTE 5 — Pensamento crítico em Inteligência Artificial
## 5.1 LEIA — Métrica alta significa solução pronta?

Não. Um resultado obtido em um conjunto de dados educacional não comprova desempenho em situações reais. É necessário considerar **representatividade dos dados**, **tipos de erro**, **generalização**, **vieses**, **reprodutibilidade** e **validação externa**.

Nesta atividade, o contexto médico é apenas o exemplo de dados. O foco da disciplina é **compreender o processo de construção e avaliação de sistemas de IA**.

## 5.2 RESPONDA E CORRIJA — Limitações e comunicação científica
Selecione as alternativas e confira suas explicações.

In [ ]:
questionario('Seção 5 — Reflexão crítica',[
 {'id':'Q14','texto':'Um modelo com alta acurácia neste dataset pode ser usado diretamente em pacientes?','opcoes':['Sim, sem qualquer outra análise','Não; seriam necessárias validações e avaliações clínicas apropriadas','Sim, porque AutoML garante segurança'],'pontos':1.0},
 {'id':'Q15','texto':'Qual atitude é mais adequada ao comunicar o resultado?','opcoes':['Informar apenas a maior acurácia','Descrever dados, separação, métricas, limitações e método','Afirmar que o AutoML nunca erra'],'pontos':1.0}
])


# Gabarito comentado — respostas

As respostas conceituais das questões Q1–Q10 e Q12–Q15 estão preenchidas abaixo. A **Q11 é variável**: depende da acurácia obtida na execução do notebook. Compare `acc_auto` com `acc_base`: marque AutoML se for maior, Convencional se for menor e Empate se forem iguais (considerando os valores exibidos).

| Questão | Resposta correta |
|---|---|
| Q1 | Uma amostra com suas medidas |\n| Q2 | Classificação binária |\n| Q3 | 30 |\n| Q4 | Para estimar o desempenho em dados não vistos |\n| Q5 | Proporção total de previsões corretas |\n| Q6 | Para observar erros e acertos por classe |\n| Q7 | Seleção automática de algoritmo e hiperparâmetros |\n| Q8 | O conjunto de treino, com validação interna |\n| Q9 | Não, orçamento e ambiente podem influenciar |\n| Q10 | O AutoML dispõe de cerca de 45 segundos para a busca |\n| Q12 | Não; é preciso considerar incerteza, contexto e validação adicional |\n| Q13 | Para garantir condições de avaliação comparáveis |\n| Q14 | Não; seriam necessárias validações e avaliações clínicas apropriadas |\n| Q15 | Descrever dados, separação, métricas, limitações e método |\n
| Q11 | Depende dos resultados: maior acurácia no teste; empate se forem iguais. |

**Observação:** execute as células na ordem. O nome completo deve ser preenchido na etapa de entrega.


In [ ]:
# Preenchimento automático das respostas conceituais.
# Execute esta célula depois de executar as partes do notebook.
RESPOSTAS.update({'Q1': 'Uma amostra com suas medidas', 'Q2': 'Classificação binária', 'Q3': '30', 'Q4': 'Para estimar o desempenho em dados não vistos', 'Q5': 'Proporção total de previsões corretas', 'Q6': 'Para observar erros e acertos por classe', 'Q7': 'Seleção automática de algoritmo e hiperparâmetros', 'Q8': 'O conjunto de treino, com validação interna', 'Q9': 'Não, orçamento e ambiente podem influenciar', 'Q10': 'O AutoML dispõe de cerca de 45 segundos para a busca', 'Q12': 'Não; é preciso considerar incerteza, contexto e validação adicional', 'Q13': 'Para garantir condições de avaliação comparáveis', 'Q14': 'Não; seriam necessárias validações e avaliações clínicas apropriadas', 'Q15': 'Descrever dados, separação, métricas, limitações e método'})
if globals().get('MODELO_OK', False):
    if acc_auto > acc_base:
        RESPOSTAS['Q11'] = 'AutoML'
    elif acc_auto < acc_base:
        RESPOSTAS['Q11'] = 'Convencional'
    else:
        RESPOSTAS['Q11'] = 'Empate'
    print('Respostas conceituais preenchidas. Q11:', RESPOSTAS['Q11'])
else:
    print('Respostas conceituais preenchidas. Execute a Parte 3 e a comparação para definir Q11.')


## Entrega
1. Salve as respostas em cada seção.
2. Preencha o nome completo e gere o JSON.
3. Envie o JSON ao professor. A nota será calculada em ambiente separado.


In [ ]:
nome=widgets.Text(description='Aluno(a):',placeholder='Nome completo',layout=widgets.Layout(width='80%'))
saida_final=widgets.Output()
def gerar(_):
    with saida_final:
        clear_output()
        ids=[f'Q{i}' for i in range(1,16)]
        faltantes=[i for i in ids if not RESPOSTAS.get(i)]
        if not nome.value.strip():
            print('Preencha seu nome completo.');return
        if faltantes:
            print('Faltam respostas:', ', '.join(faltantes));return
        registro={'aluno':nome.value.strip(),'data_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'respostas':{i:RESPOSTAS[i] for i in ids},'acuracia_convencional':float(acc_base) if 'acc_base' in globals() else None,'acuracia_automl':float(acc_auto) if 'acc_auto' in globals() else None,'algoritmo_automl':str(automl.best_estimator) if globals().get('MODELO_OK',False) else None}
        arquivo='entrega_automl.json'
        with open(arquivo,'w',encoding='utf-8') as f:json.dump(registro,f,ensure_ascii=False,indent=2)
        print('Entrega gerada. A correção e a nota serão divulgadas pelo professor.')
        try:
            from google.colab import files
            files.download(arquivo)
        except ImportError: print('Baixe o arquivo pelo explorador de arquivos.')
botao_final=widgets.Button(description='Gerar arquivo para entrega',button_style='primary',icon='download')
botao_final.on_click(gerar)
display(widgets.VBox([nome,botao_final,saida_final]))


## Orientações de entrega
Entregue o arquivo JSON ao professor. O gabarito e a correção ficam em um notebook separado, não distribuído à turma. A nota não é calculada no Colab do aluno.
